第一步：明确两个分布

对于单 logit 的二分类模型，输出实数 $z$。经过 Sigmoid 激活后得到概率 $p = \sigma(z)$，构成伯努利分布。

* **Teacher 的分布（目标分布 $P$）**：$p_t = \sigma(z_t / \tau)$，即 $P(1)=p_t$，$P(0)=1-p_t$。
* **Student 的分布（预测分布 $Q$）**：$p_s = \sigma(z_s / \tau)$，即 $Q(1)=p_s$，$Q(0)=1-p_s$。

第二步：$D_{KL}(P\|Q) = p_t\log\frac{p_t}{p_s} + (1-p_t)\log\frac{1-p_t}{1-p_s}$

拆开对数，前一项 $p_t\log p_t + (1-p_t)\log(1-p_t)$ 是 teacher 的熵，冻结时与 $p_s$ 无关，求导为 0。

第三步：省略常数项后 $-\big[p_t\log p_s + (1-p_t)\log(1-p_s)\big] = \mathrm{BCE}(p_t, p_s)$。

即：**二分类的 KL 蒸馏损失 = 以 teacher 概率为软标签的 BCE**，因为 Bernoulli 只有两个取值，softmax 的 $T^2$ 缩放在这里体现为 logit 除以 $T$ 后再算 BCE。

In [1]:
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, TensorDataset

root = Path(r"C:\Users\hank\Desktop\Research Reproduction\dianxian\self")
if str(root) not in sys.path:
    sys.path.append(str(root))

import tools as t
import models as m

log_dir = root / "train_log"
weight = root / "weight"

seed = 3
val = (0.1, 0.2)      # 自编码器 / 学生验证集比例
method = "LOOCV"
batch = 32
epochs = 100
patience = 15
alpha = 0.5           # 蒸馏系数（先用手给的，不扫）
T = 2                 # 温度
elec_names = ['Fp1-F7', 'Fp2-F8', 'T7-P7', 'T8-P8', 'P7-O1', 'P8-O2']

C:\Users\hank\Desktop\Research Reproduction\dianxian\self\models.py:259: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = mne.channels.make_standard_montage(montage_name)


$$\mathcal{L}_{\text{FGL}} =
\underbrace{\alpha \cdot \mathrm{BCE}\big(\sigma(S_\theta(x)),\; y\big)}_{\text{任务损失}}
+ \underbrace{(1-\alpha)\,\tau^2 \cdot \mathrm{BCE}\left(\sigma\left(\frac{S_\theta(x)}{\tau}\right),\; \sigma\left(\frac{T_\phi(x)}{\tau}\right)\right)}_{\text{蒸馏损失（Bernoulli KL）}}$$

$x$ 为同一个窗口，teacher 和 student 吃同一份 $x$（CHBMIT 口径无时间错位）；$y\in\{0,1\}$ 是 student 的真实标签；$T_\phi$ 冻结。

In [2]:
def distill_loss(student_logits, y_student, teacher_logits, alpha=0.5, T=2):
    """所有输出都是 logits；二分类单 logit，用 BCE + 温度后的 BCE 代替 KL"""
    ce = F.binary_cross_entropy_with_logits(student_logits.squeeze(-1), y_student.squeeze(-1))
    with torch.no_grad():
        p_t = torch.sigmoid(teacher_logits / T).squeeze(-1)
    distill = F.binary_cross_entropy_with_logits((student_logits / T).squeeze(-1), p_t) * T ** 2
    return alpha * ce + (1 - alpha) * distill

In [3]:
windows, meta = t.preprocess(r"D:\data\dianxian\wang2021\chb22", SPH=2*60, SOP=30*60,
                             interictal_thr=2*60*60, window_length=4, window_overlap=2,
                             elec_names=elec_names)

[边界] chb22 末尾不足整窗: ['chb22_11.edf(丢 1.0s)']
[chb22] 原始窗 55774
[chb22]   label=0 39544
[chb22]   label=1 2366
[chb22]   label=2 96
[chb22]   label=3 13768
[chb22]   excluded(overlap) 22
[chb22]   excluded(sph) 174
[chb22]   excluded(exclusion) 13572
[chb22] usable 42006 / excluded 13768
[chb22] Fp1-F7 窗 55774
[chb22] Fp2-F8 窗 55774
[chb22] T7-P7 窗 55774
[chb22] T8-P8 窗 55774
[chb22] P7-O1 窗 55774
[chb22] P8-O2 窗 55774


In [4]:
student_data_set = t.train_val_test(elec_names, meta, windows, seed, val=val, method="LOOCV")
# 教师的折向学生看齐：同一折、同一边（train/val/test），ictal 与间期 1:1
teacher_data_set = t.teacher_from_student(elec_names, meta, windows, student_data_set[0], seed)

# teacher_from_student 已跳过凑不出平衡数据的折，学生也要同步剔掉（按 seizure_id，顺序不变）
keep = {d["seizure_id"] for d in teacher_data_set}
student_folds = [[f for f in elec if f["seizure_id"] in keep] for elec in student_data_set]

# 统一归一化参数：每患者每折一套，统计池排除 teacher 与 student 的 val/test 窗（含其发作的 ictal/preictal）
norm_stats = [t.fold_norm_stats(meta, windows, elec_names, s, d)
              for s, d in zip(student_folds[0], teacher_data_set)]
print("折数", len(teacher_data_set), [d["seizure_id"] for d in teacher_data_set])

[划分] method=LOOCV 共 3 折
[划分] seizure  train(pre/int) val(pre/int)   test(pre/int)  autoenc(train/val)
[划分] S0       570/570        898/898        898/898        47886/5321      
[划分] S1       570/570        898/898        898/898        47886/5321      
[划分] S2       898/898        898/898        570/570        44646/4961      
[划分] method=LOOCV 共 3 折
[划分] seizure  train(pre/int) val(pre/int)   test(pre/int)  autoenc(train/val)
[划分] S0       570/570        898/898        898/898        47886/5321      
[划分] S1       570/570        898/898        898/898        47886/5321      
[划分] S2       898/898        898/898        570/570        44646/4961      
[划分] method=LOOCV 共 3 折
[划分] seizure  train(pre/int) val(pre/int)   test(pre/int)  autoenc(train/val)
[划分] S0       570/570        898/898        898/898        47886/5321      
[划分] S1       570/570        898/898        898/898        47886/5321      
[划分] S2       898/898        898/898        570/570        44646/4961      
[划分] metho

### 每折：先训 teacher（六通道），冻结后逐电极蒸馏 student

teacher 和 student 共用该折的 `mean, std`（同一套数值），所以两边的输入尺度一致，KL 项不会被尺度差异污染。

In [5]:
loss_fuction = nn.BCEWithLogitsLoss()
now = datetime.now().strftime("%y%m_%d_%H%M%S")
rows = []

for fold, d in enumerate(teacher_data_set):
    mean, std = norm_stats[fold]

    # ---- teacher：六通道 ictal vs 间期 ----
    teacher_train_dataloader, teacher_val_dataloader = [
        DataLoader(TensorDataset(t.norm_wins(d[key + "_wins"], mean, std), d["y_" + key]),
                   batch_size=batch, shuffle=(key == "train"))
        for key in ("train", "val")]

    teacher = m.Detection().to("cuda")
    optimizer_t = torch.optim.Adam(teacher.parameters(), lr=1e-3)
    schedule_t = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_t, mode="min", factor=0.5, patience=5, min_lr=1e-6)
    best_val_loss, best_t, patience_count = np.inf, {}, 0

    for epoch in range(epochs):
        teacher.train()
        for x, y in teacher_train_dataloader:
            x, y = x.to("cuda"), y.to("cuda")
            optimizer_t.zero_grad()
            loss = loss_fuction(teacher(x), y)
            loss.backward()
            optimizer_t.step()

        teacher.eval()
        avg_val_loss = 0
        with torch.no_grad():
            for x, y in teacher_val_dataloader:
                x, y = x.to("cuda"), y.to("cuda")
                avg_val_loss += loss_fuction(teacher(x), y).item()
        avg_val_loss /= len(teacher_val_dataloader)
        schedule_t.step(avg_val_loss)

        if avg_val_loss < best_val_loss:
            best_val_loss, patience_count = avg_val_loss, 0
            best_t = {k: v.cpu().clone() for k, v in teacher.state_dict().items()}
        else:
            patience_count += 1
            if patience_count >= patience:
                break

    teacher.load_state_dict(best_t)
    teacher.eval()
    teacher.requires_grad_(False)
    torch.save(best_t, weight / f"{now}_teacher_fold{fold+1}.pth")

    # 蒸馏时 teacher 和 student 吃同一个时间窗：teacher 要六通道，按 student 该折的索引拼成 (n,6,L)
    s0 = student_folds[0][fold]
    x6 = {key: t.norm_wins(torch.tensor(
              np.stack([windows[en][s0[key + "_idx"]] for en in elec_names], axis=1),
              dtype=torch.float32), mean, std)
          for key in ("train", "val", "test")}

    # ---- student：单通道，每个电极独立一个模型 ----
    for e, elec in enumerate(elec_names):
        s = student_folds[e][fold]
        mu, sg = mean[e:e+1], std[e:e+1]
        train_dataloader, val_dataloader, test_dataloader = [
            DataLoader(TensorDataset(t.norm_wins(s[key + "_wins"], mu, sg), x6[key], s["y_" + key]),
                       batch_size=batch, shuffle=(key == "train"))
            for key in ("train", "val", "test")]

        student = m.ConvBILSTMClassify().to("cuda")
        optimizer_s = torch.optim.Adam(student.parameters(), lr=1e-4)
        schedule_s = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer_s, mode="min", factor=0.5, patience=5, min_lr=1e-5)
        best_val_loss, best_s, patience_count = np.inf, {}, 0

        for epoch in range(epochs):
            student.train()
            for x, xt, y in train_dataloader:
                x, xt, y = x.to("cuda"), xt.to("cuda"), y.to("cuda")
                optimizer_s.zero_grad()
                with torch.no_grad():
                    teacher_logits = teacher(xt)
                loss = distill_loss(student(x), y, teacher_logits, alpha=alpha, T=T)
                loss.backward()
                optimizer_s.step()

            student.eval()
            avg_val_loss = 0
            with torch.no_grad():
                for x, xt, y in val_dataloader:
                    x, xt, y = x.to("cuda"), xt.to("cuda"), y.to("cuda")
                    avg_val_loss += distill_loss(student(x), y, teacher(xt), alpha=alpha, T=T).item()
            avg_val_loss /= len(val_dataloader)
            schedule_s.step(avg_val_loss)

            if avg_val_loss < best_val_loss:
                best_val_loss, patience_count = avg_val_loss, 0
                best_s = {k: v.cpu().clone() for k, v in student.state_dict().items()}
            else:
                patience_count += 1
                if patience_count >= patience:
                    break

        student.load_state_dict(best_s)
        student.eval()
        probs, targets = [], []
        with torch.no_grad():
            for x, xt, y in test_dataloader:
                probs.extend(torch.sigmoid(student(x.to("cuda"))).cpu().numpy().ravel())
                targets.extend(y.numpy().ravel())
        probs, targets = np.array(probs), np.array(targets)
        rows.append(dict(fold=fold + 1, seizure_id=d["seizure_id"], elec=elec,
                         n_test=len(targets),
                         auc=round(float(roc_auc_score(targets, probs)), 4),
                         acc=round(float(((probs > 0.5) == targets).mean()), 4)))
        torch.save(best_s, weight / f"{now}_student_fold{fold+1}_{elec}.pth")

    del x6, teacher
    torch.cuda.empty_cache()
    print(f"fold {fold+1} seizure {d['seizure_id']} 完成")


fold 1 seizure 0 完成
fold 2 seizure 1 完成
fold 3 seizure 2 完成


In [6]:
df = pd.DataFrame(rows)
path = log_dir / f"distill_{now}.csv"
df.to_csv(path, index=False, encoding="utf-8")
print(path)
df.pivot_table(index=["fold", "seizure_id"], columns="elec", values="auc")

C:\Users\hank\Desktop\Research Reproduction\dianxian\self\train_log\distill_2611_14_184323.csv


,elec,Fp1-F7,Fp2-F8,P7-O1,P8-O2,T7-P7,T8-P8
fold,seizure_id,,,,,,
1,0,0.6019,0.6004,0.8793,0.8472,0.7929,0.8195
2,1,0.4152,0.3937,0.5753,0.4774,0.4191,0.5157
3,2,0.8925,0.8789,0.9653,0.8342,0.9607,0.9734
